In [4]:
import pickle
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML

# ── Load model, scaler & dataset ─────────────────────────────
model   = pickle.load(open('breast_cancer_model.pkl', 'rb'))
scaler  = pickle.load(open('scaler.pkl', 'rb'))
df      = pd.read_csv('breast_cancer.csv')   # 🔁 change to your filename

# ── Style ─────────────────────────────────────────────────────
style  = {'description_width': '160px'}
layout = widgets.Layout(width='420px')
int_layout = widgets.Layout(width='420px')

# ── ID Lookup ─────────────────────────────────────────────────
w_id = widgets.IntText(
    description='🔎 Enter ID:',
    style=style, layout=layout
)
btn_load = widgets.Button(
    description='📂 Load from Dataset',
    button_style='info',
    layout=widgets.Layout(width='420px', height='36px')
)

# ── Feature Widgets (1–10 range for Wisconsin dataset) ────────
w_thick   = widgets.IntSlider(description='Cl. Thickness',    min=1,max=10,value=5, style=style, layout=layout)
w_csize   = widgets.IntSlider(description='Cell Size',        min=1,max=10,value=5, style=style, layout=layout)
w_cshape  = widgets.IntSlider(description='Cell Shape',       min=1,max=10,value=5, style=style, layout=layout)
w_madh    = widgets.IntSlider(description='Marg. Adhesion',   min=1,max=10,value=5, style=style, layout=layout)
w_epith   = widgets.IntSlider(description='Epith. C. Size',   min=1,max=10,value=5, style=style, layout=layout)
w_bnuclei = widgets.IntSlider(description='Bare Nuclei',      min=1,max=10,value=5, style=style, layout=layout)
w_bchrom  = widgets.IntSlider(description='Bl. Cromatin',     min=1,max=10,value=5, style=style, layout=layout)
w_nnucl   = widgets.IntSlider(description='Normal Nucleoli',  min=1,max=10,value=5, style=style, layout=layout)
w_mitoses = widgets.IntSlider(description='Mitoses',          min=1,max=10,value=1, style=style, layout=layout)

all_widgets = [w_id,w_thick, w_csize, w_cshape, w_madh,
               w_epith, w_bnuclei, w_bchrom, w_nnucl, w_mitoses]

# ── Load row by ID ────────────────────────────────────────────
load_output = widgets.Output()

def on_load(b):
    load_output.clear_output()
    pid = w_id.value
    row = df[df['Id'] == pid]
    if row.empty:
        with load_output:
            display(HTML(f"<p style='color:red'>❌ ID <b>{pid}</b> not found in dataset.</p>"))
        return
    r = row.iloc[0]
    w_thick.value   = int(r['Cl.thickness'])
    w_csize.value   = int(r['Cell.size'])
    w_cshape.value  = int(r['Cell.shape'])
    w_madh.value    = int(r['Marg.adhesion'])
    w_epith.value   = int(r['Epith.c.size'])
    w_bnuclei.value = int(r['Bare.nuclei'])
    w_bchrom.value  = int(r['Bl.cromatin'])
    w_nnucl.value   = int(r['Normal.nucleoli'])
    w_mitoses.value = int(r['Mitoses'])
    with load_output:
        display(HTML(f"<p style='color:#86efac'>✅ ID <b>{pid}</b> loaded successfully!</p>"))

btn_load.on_click(on_load)

# ── Predict Button ────────────────────────────────────────────
btn_predict = widgets.Button(
    description='🔍 Predict',
    button_style='primary',
    layout=widgets.Layout(width='420px', height='42px')
)
predict_output = widgets.Output()

def on_predict(b):
    predict_output.clear_output()
    features = np.array([[
        w_id.value,
        w_thick.value, w_csize.value, w_cshape.value,
        w_madh.value,  w_epith.value, w_bnuclei.value,
        w_bchrom.value, w_nnucl.value, w_mitoses.value
    ]])

    features_scaled = scaler.transform(features)
    prediction      = model.predict(features_scaled)[0]
    probability     = model.predict_proba(features_scaled)[0]
    confidence      = f"{max(probability) * 100:.2f}%"

    with predict_output:
        if prediction == 1:
            display(HTML(f"""
            <div style='padding:16px; background:#7f1d1d; border:2px solid #dc2626;
                        border-radius:10px; text-align:center; margin-top:10px;'>
                <h2 style='color:#fca5a5; margin:0'>🔴 Malignant — Cancer Detected</h2>
                <p style='color:#fecaca; margin:6px 0 0'>Confidence: <b>{confidence}</b></p>
                <p style='color:#fca5a5; font-size:12px; margin:4px 0 0'>
                   ⚠️ Please consult a doctor immediately.</p>
            </div>"""))
        else:
            display(HTML(f"""
            <div style='padding:16px; background:#14532d; border:2px solid #16a34a;
                        border-radius:10px; text-align:center; margin-top:10px;'>
                <h2 style='color:#86efac; margin:0'>🟢 Benign — No Cancer Detected</h2>
                <p style='color:#bbf7d0; margin:6px 0 0'>Confidence: <b>{confidence}</b></p>
                <p style='color:#86efac; font-size:12px; margin:4px 0 0'>
                   ✅ Continue regular medical checkups.</p>
            </div>"""))

btn_predict.on_click(on_predict)

# ── Display Full UI ───────────────────────────────────────────
display(HTML("<h2 style='color:#60a5fa; margin-bottom:4px'>🏥 Breast Cancer Predictor</h2>"))
display(HTML("<p style='color:gray; margin-top:0'>Enter values manually <b>or</b> load by Patient ID</p>"))
display(HTML("<hr style='border-color:#374151'>"))

# ID Lookup section
display(HTML("<b style='color:#93c5fd'>🔎 Load Patient by ID</b>"))
display(w_id, btn_load, load_output)
display(HTML("<hr style='border-color:#374151'>"))

# Feature sliders
display(HTML("<b style='color:#93c5fd'>📋 Clinical Features (1–10)</b>"))
for w in all_widgets:
    display(w)

display(HTML("<br>"))
display(btn_predict, predict_output)

IntText(value=0, description='🔎 Enter ID:', layout=Layout(width='420px'), style=DescriptionStyle(description_w…

Button(button_style='info', description='📂 Load from Dataset', layout=Layout(height='36px', width='420px'), st…

Output()

IntText(value=0, description='🔎 Enter ID:', layout=Layout(width='420px'), style=DescriptionStyle(description_w…

IntSlider(value=5, description='Cl. Thickness', layout=Layout(width='420px'), max=10, min=1, style=SliderStyle…

...

IntSlider(value=5, description='Bl. Cromatin', layout=Layout(width='420px'), max=10, min=1, style=SliderStyle(…

IntSlider(value=5, description='Normal Nucleoli', layout=Layout(width='420px'), max=10, min=1, style=SliderSty…

IntSlider(value=1, description='Mitoses', layout=Layout(width='420px'), max=10, min=1, style=SliderStyle(descr…

Button(button_style='primary', description='🔍 Predict', layout=Layout(height='42px', width='420px'), style=But…

Output()